# **Lucas–Kanade a mano**

Flujo de trabajo:
1. Pasar la imagen a blanco y negro.
2. Aplicar filtro gaussiano (práctica 3).
3. Detectar bordes con Canny (https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html).
4. Escoger un borde (para esquina Harris).
5. Aplicar una máscara y buscar en ella los cambios de intensidad.
6. Sobel para las derivadas.

In [24]:
import numpy as np
import cv2
import os
import numpy as np
import cv2

out_dir = "output/1"
os.makedirs(out_dir, exist_ok=True)

# frame= cv2.VideoCapture(0)
frame = cv2.VideoCapture("videos/people2.mp4")

# toma la primera imagen
status, old_frame = frame.read()
print(f"Dimensiones imagen original: {old_frame.shape}")
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
cv2.imwrite(f"{out_dir}/original.jpg", old_gray)

# Aplicar suavizado Gaussiano
gauss = cv2.GaussianBlur(old_gray, (5,5), 0)
cv2.imwrite(f"{out_dir}/suavizado.jpg", gauss)

# Detectamos los bordes con Canny
canny = cv2.Canny(gauss, 50, 150)
cv2.imwrite(f"{out_dir}/canny.jpg", canny)
 
# Buscamos los contornos https://programarfacil.com/blog/vision-artificial/detector-de-bordes-canny-opencv/
(contornos,_) = cv2.findContours(canny.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contornos_img = old_gray.copy()
cv2.drawContours(contornos_img,contornos,-1,(0,0,255), 2)
cv2.imwrite(f"{out_dir}/contornos.jpg", contornos_img)

cnt = contornos[0]

# punto más arriba
top = cnt[cnt[:,:,1].argmin()][0]
#top = (120, 85)  # una coordenada -> TODO buscar esquina (p.e.: Harris)


# Función get_mask_5x5 pedida a ChatGPT para devolver la máscara 5x5 alrededor del pixel
def get_mask_5x5(img, point):
    x, y = point

    # límites de la máscara
    x1 = max(0, x - 2)
    x2 = min(img.shape[1], x + 3)

    y1 = max(0, y - 2)
    y2 = min(img.shape[0], y + 3)

    mask = img[y1:y2, x1:x2]
    return mask


intensidad_1 = get_mask_5x5(old_gray, top)

print(f"Máscara 5x5:\n{intensidad_1.shape}")

sobelx = cv2.Sobel(intensidad_1, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(intensidad_1, cv2.CV_64F, 0, 1, ksize=3)

# https://docs.opencv.org/4.x/d2/d2c/tutorial_sobel_derivatives.html
abs_grad_x = cv2.convertScaleAbs(sobelx)
abs_grad_y = cv2.convertScaleAbs(sobely)

print(f"Magnitudes en x:\n{abs_grad_x.ravel().shape}")
print(f"Magnitudes en y:\n{abs_grad_y.ravel().shape}")
matrix = np.column_stack((abs_grad_x.ravel(), abs_grad_y.ravel())) # matrix de las derivadas
print(f"matrix:\n{matrix.shape}")
cv2.waitKey(0)
cv2.destroyAllWindows()

old_frame_copy = old_frame.copy()
cv2.circle(old_frame_copy, top, 2, (0, 0, 255), -1)
cv2.imshow("punto inicial", old_frame_copy)
cv2.imwrite(f"{out_dir}/punto_inicial_frame0.jpg", old_frame_copy)

#inicializo una imagen para añadir los resultados en blanco
result2 =  np.zeros_like(old_gray)
frame_idx = 0
while status:
    if frame_idx ==2:
       break

    #captura la imagen 2
    ret, img = frame.read()
    if not ret:
       print("No hay más frames o el frame no se pudo leer.")
       break
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    intensidad_2 = get_mask_5x5(gray, top)
    
    T = intensidad_2.ravel() - intensidad_1.ravel()
    print(f"matrix.T:\n{matrix.T.shape}")
    print(f"matrix:\n{matrix.shape}")
    print(f"T:\n{T.shape}")
    diff_p = np.linalg.inv((matrix.T @ matrix)) @ matrix.T @ T
    print(f"diff_p:\n{diff_p}")
    top_arr = np.array(top, dtype=np.float32)
    print(f"old point: {top_arr}")
    new_point = top_arr + diff_p
    print(f"new_point: {new_point}")
  
    new = img.copy()
    new_point_int = (int(new_point[0]), int(new_point[1]))
    cv2.circle(new, new_point_int, 3, (0, 255, 0), -1)
    cv2.imwrite(f"{out_dir}/punto_final_frame{frame_idx}.jpg", new)
    cv2.imshow(f"punto final frame {frame_idx}", new)
  
    cv2.waitKey(0)
  
    frame_idx +=1
      
frame.release()
cv2.waitKey(0)
cv2.destroyAllWindows()

Dimensiones imagen original: (720, 1280, 3)
Máscara 5x5:
(5, 5)
Magnitudes en x:
(25,)
Magnitudes en y:
(25,)
matrix:
(25, 2)
matrix.T:
(2, 25)
matrix:
(25, 2)
T:
(25,)
diff_p:
[1416.41504854  914.40533981]
old point: [822. 698.]
new_point: [2238.41504854 1612.40533981]
matrix.T:
(2, 25)
matrix:
(25, 2)
T:
(25,)
diff_p:
[1145.63280166  541.35818308]
old point: [822. 698.]
new_point: [1967.63280166 1239.35818308]


El código anterior realiza un movimiento muy brusco. ERROR

Correcciones del código anterior:
- quitar la conversión de la escala absoluta de los gradientes para que coja el signo.
- cambiar `T = intensidad_2.ravel() - intensidad_1.ravel()` a   `T = intensidad_1.ravel() - intensidad_2ravel()` siguiendo las fórmulas de las diapositivas.

In [25]:
import numpy as np
import cv2
import os
import numpy as np
import cv2

out_dir = "output/2"
os.makedirs(out_dir, exist_ok=True)

# frame= cv2.VideoCapture(0)
frame = cv2.VideoCapture("videos/people2.mp4")

# toma la primera imagen
status, old_frame = frame.read()
print(f"Dimensiones imagen original: {old_frame.shape}")
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
cv2.imwrite(f"{out_dir}/original.jpg", old_gray)

# Aplicar suavizado Gaussiano
gauss = cv2.GaussianBlur(old_gray, (5,5), 0)
cv2.imwrite(f"{out_dir}/suavizado.jpg", gauss)

# Detectamos los bordes con Canny
canny = cv2.Canny(gauss, 50, 150)
cv2.imwrite(f"{out_dir}/canny.jpg", canny)
 
# Buscamos los contornos https://programarfacil.com/blog/vision-artificial/detector-de-bordes-canny-opencv/
(contornos,_) = cv2.findContours(canny.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contornos_img = old_gray.copy()
cv2.drawContours(contornos_img,contornos,-1,(0,0,255), 2)
cv2.imwrite(f"{out_dir}/contornos.jpg", contornos_img)

cnt = contornos[0]

# punto más arriba
top = cnt[cnt[:,:,1].argmin()][0]
#top = (120, 85)  # una coordenada -> TODO buscar esquina (p.e.: Harris)


# Función get_mask_5x5 pedida a ChatGPT para devolver la máscara 5x5 alrededor del pixel
def get_mask_5x5(img, point):
    x, y = point

    # límites de la máscara
    x1 = max(0, x - 2)
    x2 = min(img.shape[1], x + 3)

    y1 = max(0, y - 2)
    y2 = min(img.shape[0], y + 3)

    mask = img[y1:y2, x1:x2]
    return mask


intensidad_1 = get_mask_5x5(old_gray, top)

print(f"Máscara 5x5:\n{intensidad_1.shape}")

sobelx = cv2.Sobel(intensidad_1, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(intensidad_1, cv2.CV_64F, 0, 1, ksize=3)

print(f"sobelx:\n{sobelx.ravel().shape}")
print(f"sobely:\n{sobely.ravel().shape}")
matrix = np.column_stack((sobelx.ravel(), sobely.ravel())) # matrix de las derivadas
print(f"matrix:\n{matrix.shape}")
cv2.waitKey(0)
cv2.destroyAllWindows()

old_frame_copy = old_frame.copy()
cv2.circle(old_frame_copy, top, 2, (0, 0, 255), -1)
cv2.imshow("punto inicial", old_frame_copy)
cv2.imwrite(f"{out_dir}/punto_inicial_frame0.jpg", old_frame_copy)

#inicializo una imagen para añadir los resultados en blanco
result2 =  np.zeros_like(old_gray)
frame_idx = 0
while status:
    if frame_idx ==2:
       break

    #captura la imagen 2
    ret, img = frame.read()
    if not ret:
       print("No hay más frames o el frame no se pudo leer.")
       break
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    intensidad_2 = get_mask_5x5(gray, top)
    
    T = intensidad_1.ravel() - intensidad_2.ravel()
    print(f"matrix.T:\n{matrix.T.shape}")
    print(f"matrix:\n{matrix.shape}")
    print(f"T:\n{T.shape}")
    diff_p = np.linalg.inv((matrix.T @ matrix)) @ matrix.T @ T
    print(f"diff_p:\n{diff_p}")
    top_arr = np.array(top, dtype=np.float32)
    print(f"old point: {top_arr}")
    new_point = top_arr + diff_p
    print(f"new_point: {new_point}")
  
    new = img.copy()
    new_point_int = (int(new_point[0]), int(new_point[1]))
    cv2.circle(new, new_point_int, 3, (0, 255, 0), -1)
    cv2.imwrite(f"{out_dir}/punto_final_frame{frame_idx}.jpg", new)
    cv2.imshow(f"punto final frame {frame_idx}", new)
  
    cv2.waitKey(0)
  
    frame_idx +=1
      
frame.release()
cv2.waitKey(0)
cv2.destroyAllWindows()

Dimensiones imagen original: (720, 1280, 3)
Máscara 5x5:
(5, 5)
sobelx:
(25,)
sobely:
(25,)
matrix:
(25, 2)
matrix.T:
(2, 25)
matrix:
(25, 2)
T:
(25,)
diff_p:
[0.29455527 0.45971541]
old point: [822. 698.]
new_point: [822.29455527 698.45971541]
matrix.T:
(2, 25)
matrix:
(25, 2)
T:
(25,)
diff_p:
[0.34675559 0.7173839 ]
old point: [822. 698.]
new_point: [822.34675559 698.7173839 ]


Ahora falta actualizar la posición del punto:

In [ ]:
import numpy as np
import cv2
import os
import numpy as np
import cv2

out_dir = "output/3"
os.makedirs(out_dir, exist_ok=True)

# frame= cv2.VideoCapture(0)
frame = cv2.VideoCapture("videos/people2.mp4")

# toma la primera imagen
status, old_frame = frame.read()
print(f"Dimensiones imagen original: {old_frame.shape}")
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
cv2.imwrite(f"{out_dir}/original.jpg", old_gray)

# Aplicar suavizado Gaussiano
gauss = cv2.GaussianBlur(old_gray, (5,5), 0)
cv2.imwrite(f"{out_dir}/suavizado.jpg", gauss)

# Detectamos los bordes con Canny
canny = cv2.Canny(gauss, 50, 150)
cv2.imwrite(f"{out_dir}/canny.jpg", canny)
 
# Buscamos los contornos https://programarfacil.com/blog/vision-artificial/detector-de-bordes-canny-opencv/
(contornos,_) = cv2.findContours(canny.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contornos_img = old_gray.copy()
cv2.drawContours(contornos_img,contornos,-1,(0,0,255), 2)
cv2.imwrite(f"{out_dir}/contornos.jpg", contornos_img)

cnt = contornos[0]

# punto más arriba
top = cnt[cnt[:,:,1].argmin()][0]
#top = (120, 85)  # una coordenada -> TODO buscar esquina (p.e.: Harris)


# Función get_mask_5x5 pedida a ChatGPT para devolver la máscara 5x5 alrededor del pixel
def get_mask_5x5(img, point):
    x, y = point

    # límites de la máscara
    x1 = max(0, x - 2)
    x2 = min(img.shape[1], x + 3)

    y1 = max(0, y - 2)
    y2 = min(img.shape[0], y + 3)

    mask = img[y1:y2, x1:x2]
    return mask

#inicializo una imagen para añadir los resultados en blanco
result2 =  np.zeros_like(old_gray)
frame_idx = 0
punto_inicial = top

while status:
    if frame_idx ==2:
       break

    intensidad_1 = get_mask_5x5(old_gray, punto_inicial)
  
    print(f"Máscara 5x5:\n{intensidad_1.shape}")
  
    sobelx = cv2.Sobel(intensidad_1, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(intensidad_1, cv2.CV_64F, 0, 1, ksize=3)
  
    print(f"sobelx:\n{sobelx.ravel().shape}")
    print(f"sobely:\n{sobely.ravel().shape}")
    matrix = np.column_stack((sobelx.ravel(), sobely.ravel())) # matrix de las derivadas
    print(f"matrix:\n{matrix.shape}")
    cv2.waitKey(0)
    cv2.destroyAllWindows()
  
    old_frame_copy = old_frame.copy()
    cv2.circle(old_frame_copy, punto_inicial, 2, (0, 0, 255), -1)
    cv2.imshow(f"punto inicial frame {frame_idx}", old_frame_copy)
  

    #captura la imagen 2
    ret, img = frame.read()
    if not ret:
       print("No hay más frames o el frame no se pudo leer.")
       break
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    intensidad_2 = get_mask_5x5(gray, punto_inicial)
    
    T = intensidad_1.ravel() - intensidad_2.ravel()
    print(f"matrix.T:\n{matrix.T.shape}")
    print(f"matrix:\n{matrix.shape}")
    print(f"T:\n{T.shape}")
    diff_p = np.linalg.inv((matrix.T @ matrix)) @ matrix.T @ T
    print(f"diff_p:\n{diff_p}")
    top_arr = np.array(punto_inicial, dtype=np.float32)
    print(f"old point: {top_arr}")
    new_point = top_arr + diff_p
    print(f"new_point: {new_point}")
  
    new = img.copy()
    new_point_int = (int(new_point[0]), int(new_point[1]))
    cv2.circle(new, new_point_int, 2, (0, 255, 0), -1)
    cv2.imwrite(f"{out_dir}/punto_final_frame{frame_idx}.jpg", new)
    cv2.imshow(f"punto final frame {frame_idx}", new)
  
    cv2.waitKey(0)
    
    punto_inicial = new_point_int
    frame_idx +=1
      
frame.release()
cv2.waitKey(0)
cv2.destroyAllWindows()

Dimensiones imagen original: (720, 1280, 3)
Máscara 5x5:
(5, 5)
sobelx:
(25,)
sobely:
(25,)
matrix:
(25, 2)
matrix.T:
(2, 25)
matrix:
(25, 2)
T:
(25,)
diff_p:
[0.29455527 0.45971541]
old point: [822. 698.]
new_point: [822.29455527 698.45971541]
Máscara 5x5:
(5, 5)
sobelx:
(25,)
sobely:
(25,)
matrix:
(25, 2)
matrix.T:
(2, 25)
matrix:
(25, 2)
T:
(25,)
diff_p:
[0.34675559 0.7173839 ]
old point: [822. 698.]
new_point: [822.34675559 698.7173839 ]


Esquinas con Harris:

In [27]:
import numpy as np
import cv2
import os
import numpy as np
import cv2

out_dir = "output/4"
os.makedirs(out_dir, exist_ok=True)

# frame= cv2.VideoCapture(0)
frame = cv2.VideoCapture("videos/people2.mp4")

# toma la primera imagen
status, old_frame = frame.read()
print(f"Dimensiones imagen original: {old_frame.shape}")
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
cv2.imwrite(f"{out_dir}/original.jpg", old_gray)

# Aplicar suavizado Gaussiano
gauss = cv2.GaussianBlur(old_gray, (5,5), 0)
cv2.imwrite(f"{out_dir}/suavizado.jpg", gauss)

# Detectamos los bordes con Canny
canny = cv2.Canny(gauss, 50, 150)
cv2.imwrite(f"{out_dir}/canny.jpg", canny)
 
# Buscamos los contornos https://programarfacil.com/blog/vision-artificial/detector-de-bordes-canny-opencv/
(contornos,_) = cv2.findContours(canny.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contornos_img = old_gray.copy()
cv2.drawContours(contornos_img,contornos,-1,(0,0,255), 2)
cv2.imwrite(f"{out_dir}/contornos.jpg", contornos_img)

cnt = contornos[0]


# Función get_mask_5x5 pedida a ChatGPT para devolver la máscara 5x5 alrededor del pixel
def get_mask_5x5(img, point):
    x, y = point

    # límites de la máscara
    x1 = max(0, x - 2)
    x2 = min(img.shape[1], x + 3)

    y1 = max(0, y - 2)
    y2 = min(img.shape[0], y + 3)

    mask = img[y1:y2, x1:x2]
    return mask

#inicializo una imagen para añadir los resultados en blanco
result2 =  np.zeros_like(old_gray)
frame_idx = 0
old_frame_copy = old_frame.copy()
# Detectar esquinas con Harris
# https://programacionpython80889555.wordpress.com/2025/09/17/deteccion-de-esquinas-en-python-con-opencv-y-numpy/
dst = cv2.cornerHarris(canny, 2, 3, 0, 4)
dst = cv2.dilate(dst, None)
img2 = old_frame.copy()
esquinas = []
# https://acodigo.blogspot.com/2017/07/deteccion-de-esquinas-con-opencv.html
for y in range(0, img2.shape[0]):
    for x in range(0, img2.shape[1]):
        if dst.item(y, x) > 0.01 * dst.max():
            esquinas.append((x, y))
punto_inicial = esquinas[round(len(esquinas)/2)]

while status:
    if frame_idx ==2:
       break

    intensidad_1 = get_mask_5x5(old_gray, punto_inicial)
  
    print(f"Máscara 5x5:\n{intensidad_1.shape}")
  
    sobelx = cv2.Sobel(intensidad_1, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(intensidad_1, cv2.CV_64F, 0, 1, ksize=3)
  
    print(f"sobelx:\n{sobelx.ravel().shape}")
    print(f"sobely:\n{sobely.ravel().shape}")
    matrix = np.column_stack((sobelx.ravel(), sobely.ravel())) # matrix de las derivadas
    print(f"matrix:\n{matrix.shape}")
    cv2.waitKey(0)
    cv2.destroyAllWindows()
  
    old_frame_copy = old_frame.copy()
    cv2.circle(old_frame_copy, punto_inicial, 2, (0, 0, 255), -1)
    cv2.imshow(f"punto inicial frame {frame_idx}", old_frame_copy)
  

    #captura la imagen 2
    ret, img = frame.read()
    if not ret:
       print("No hay más frames o el frame no se pudo leer.")
       break
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    intensidad_2 = get_mask_5x5(gray, punto_inicial)
    
    T = intensidad_1.ravel() - intensidad_2.ravel()
    print(f"matrix.T:\n{matrix.T.shape}")
    print(f"matrix:\n{matrix.shape}")
    print(f"T:\n{T.shape}")
    diff_p = np.linalg.inv((matrix.T @ matrix)) @ matrix.T @ T
    print(f"diff_p:\n{diff_p}")
    top_arr = np.array(punto_inicial, dtype=np.float32)
    print(f"old point: {top_arr}")
    new_point = top_arr + diff_p
    print(f"new_point: {new_point}")
  
    new = img.copy()
    new2 = img.copy()
    new_point_int = (int(new_point[0]), int(new_point[1]))
    cv2.circle(new, new_point_int, 2, (0, 255, 0), -1)
    cv2.imwrite(f"{out_dir}/punto_final_frame{frame_idx}.jpg", new)
    cv2.imshow(f"punto final frame {frame_idx}", new)
  
    cv2.waitKey(0)
    
    punto_inicial = new_point_int
    frame_idx +=1
      
frame.release()
cv2.waitKey(0)
cv2.destroyAllWindows()

Dimensiones imagen original: (720, 1280, 3)
Máscara 5x5:
(5, 5)
sobelx:
(25,)
sobely:
(25,)
matrix:
(25, 2)
matrix.T:
(2, 25)
matrix:
(25, 2)
T:
(25,)
diff_p:
[0.26552115 0.17943085]
old point: [1213.  182.]
new_point: [1213.26552115  182.17943085]
Máscara 5x5:
(5, 5)
sobelx:
(25,)
sobely:
(25,)
matrix:
(25, 2)
matrix.T:
(2, 25)
matrix:
(25, 2)
T:
(25,)
diff_p:
[0.2535857  0.22856975]
old point: [1213.  182.]
new_point: [1213.2535857   182.22856975]


Habría que valorar si utilizar Harris o utilizar los contornos.